# M3-B1 — Exploration des 3 sources Acerox

> Notebook d'**exploration rapide** — pas d'EDA fouillée, juste assez pour
> remplir l'inventaire de la note d'identification.

Auteur·rice : `<prénom>` — Date : `<date>`

**Règles** :
- Pas de transformation (juste lecture, `info`, `head`, `describe`)
- Une cellule markdown par source — qu'est-ce que tu observes ?
- Trace les **risques** et **questions** qui émergent pour l'`identification_sources.md`

In [1]:
from pathlib import Path
import json

import pandas as pd

DATA_DIR = Path("../data")

## Source 1 — Capteurs IoT (CSV)

In [2]:
df_iot = pd.read_csv(DATA_DIR / "capteurs_iot.csv")
df_iot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51000 entries, 0 to 50999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      51000 non-null  object 
 1   site           51000 non-null  object 
 2   line_id        51000 non-null  int64  
 3   sensor_id      51000 non-null  object 
 4   temperature_c  51000 non-null  float64
 5   vibration_mms  50251 non-null  float64
 6   debit_uh       51000 non-null  float64
dtypes: float64(3), int64(1), object(3)
memory usage: 2.7+ MB


In [3]:
# Aperçu
display(df_iot.head())

# Statistiques descriptives (numériques + catégorielles)
display(df_iot.describe(include="all").transpose())

# Volume
print(f"Lignes: {len(df_iot):,}")
print(f"Colonnes: {df_iot.shape[1]}")
mem_mb_iot = df_iot.memory_usage(deep=True).sum() / (1024**2)
print(f"Mémoire estimée en RAM: {mem_mb_iot:.2f} Mo")

# Défauts manifestes
display(pd.DataFrame({
    "dtype": df_iot.dtypes.astype(str),
    "missing": df_iot.isna().sum(),
    "missing_pct": (df_iot.isna().mean() * 100).round(2),
    "nunique": df_iot.nunique(dropna=True),
}).sort_values(["missing_pct", "nunique"], ascending=[False, True]))

# Repérage heuristique de champs potentiellement sensibles (RGPD)
rgpd_keywords = ["name", "nom", "prenom", "first_name", "last_name", "email", "mail", "phone", "tel", "address", "adresse", "city", "postal", "zip", "employee", "worker", "ouvrier", "operator", "user", "id"]
potential_sensitive_iot = [
    c for c in df_iot.columns
    if any(k in c.lower() for k in rgpd_keywords)
]
print("Colonnes potentiellement sensibles:", potential_sensitive_iot if potential_sensitive_iot else "Aucune détectée par nom de colonne")

# Défauts spécifiques fréquents
dup_iot = df_iot.duplicated().sum()
print(f"Doublons exacts: {dup_iot:,}")

,timestamp,site,line_id,sensor_id,temperature_c,vibration_mms,debit_uh
0,2026-04-14T19:21:43,Lyon,1,SLYO-L1-T01,77.92,5.539793,101.27
1,2026-04-27T02:47:12,Lyon,1,SLYO-L1-T01,70.58,3.361715,110.19
2,2026-04-13T18:18:50,Saint-Etienne,1,SSAI-L1-T01,62.37,4.019277,111.28
3,2026-04-05T10:34:03,Roubaix,2,SROU-L2-T01,66.17,4.922531,123.93
4,2026-04-20T10:18:07,Saint-Etienne,3,SSAI-L3-T01,55.56,1.643043,101.40


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,51000,49505,2026-04-08T16:39:30,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site,51000,3,Roubaix,20570,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line_id,51000.0,NaN,NaN,NaN,2.005882,1.033472,1.0,1.0,2.0,3.0,4.0
sensor_id,51000,8,SLYO-L1-T01,10182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
temperature_c,51000.0,NaN,NaN,NaN,73.717034,27.005577,26.47,60.2375,66.09,72.84,160.0
vibration_mms,50251.0,NaN,NaN,NaN,4.831502,2.685963,0.0,3.302777,4.187726,5.171534,12.0
debit_uh,51000.0,NaN,NaN,NaN,110.050188,11.832749,80.0,102.0,110.1,118.05,150.0


Lignes: 51,000
Colonnes: 7
Mémoire estimée en RAM: 10.59 Mo


,dtype,missing,missing_pct,nunique
vibration_mms,float64,749,1.47,44168
site,object,0,0.00,3
line_id,int64,0,0.00,4
sensor_id,object,0,0.00,8
debit_uh,float64,0,0.00,5842
temperature_c,float64,0,0.00,6132
timestamp,object,0,0.00,49505


Colonnes potentiellement sensibles: ['line_id', 'sensor_id']
Doublons exacts: 1,000


> **Observations** :
>
> - Volume : **51 000 lignes**, 7 colonnes, ~**10,59 Mo** en RAM.
> - Période : timestamps au format texte (`object`), forte granularité temporelle.
> - Qualité observée : `vibration_mms` a **749 manquants (1,47%)** ; **1 000 doublons exacts**.
> - Risques RGPD : pas de PII explicite, mais `sensor_id` et `line_id` sont des identifiants techniques potentiellement indirectement identifiants en croisement.
> - Pertinence métier : très bonne pour le suivi process (température, vibration, débit), utile pour corrélation incidents/performance.
> - Question pour Sébastien : les doublons correspondent-ils à des replays techniques ou à une vraie duplication de collecte ?

## Source 2 — ERP (JSON)

In [4]:
with (DATA_DIR / "erp_export.json").open() as f:
    orders = json.load(f)
df_erp = pd.DataFrame(orders)
df_erp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   ordre_id         2000 non-null   int64 
 1   produit_ref      2000 non-null   object
 2   site             2000 non-null   object
 3   line_id          2000 non-null   int64 
 4   date_lancement   2000 non-null   object
 5   date_fin_prevue  2000 non-null   object
 6   statut           2000 non-null   object
 7   ouvrier_id       1891 non-null   object
 8   quantite_kg      2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB


In [5]:
# Aperçu
display(df_erp.head())

# Statistiques descriptives
display(df_erp.describe(include="all").transpose())

# Volume
print(f"Lignes: {len(df_erp):,}")
print(f"Colonnes: {df_erp.shape[1]}")
mem_mb_erp = df_erp.memory_usage(deep=True).sum() / (1024**2)
print(f"Mémoire estimée en RAM: {mem_mb_erp:.2f} Mo")

# Défauts manifestes
quality_erp = pd.DataFrame({
    "dtype": df_erp.dtypes.astype(str),
    "missing": df_erp.isna().sum(),
    "missing_pct": (df_erp.isna().mean() * 100).round(2),
    "nunique": df_erp.nunique(dropna=True),
}).sort_values(["missing_pct", "nunique"], ascending=[False, True])
display(quality_erp)

# Exemple ciblé demandé: distribution du statut si présent
status_col = next((c for c in df_erp.columns if "stat" in c.lower()), None)
if status_col:
    print(f"\nRépartition de {status_col}:")
    display(df_erp[status_col].value_counts(dropna=False))
else:
    print("Aucune colonne de statut détectée automatiquement.")

# Repérage heuristique de champs potentiellement sensibles (RGPD)
rgpd_keywords = ["name", "nom", "prenom", "first_name", "last_name", "email", "mail", "phone", "tel", "address", "adresse", "city", "postal", "zip", "employee", "worker", "ouvrier", "salary", "iban", "siret", "birth", "date_naissance", "id"]
potential_sensitive_erp = [
    c for c in df_erp.columns
    if any(k in c.lower() for k in rgpd_keywords)
]
print("Colonnes potentiellement sensibles:", potential_sensitive_erp if potential_sensitive_erp else "Aucune détectée par nom de colonne")

if "ouvrier_id" in df_erp.columns:
    missing_worker = df_erp["ouvrier_id"].isna().mean() * 100
    print(f"Taux de manquants sur ouvrier_id: {missing_worker:.2f}%")

,ordre_id,produit_ref,site,line_id,date_lancement,date_fin_prevue,statut,ouvrier_id,quantite_kg
0,100000,ALU-T1-22,Roubaix,3,2026-04-01T22:21:08,2026-04-02T23:21:08,suspendu,EMP-5317,3221
1,100001,INOX-316-4,Saint-Etienne,1,2026-04-26T14:52:52,2026-04-28T15:52:52,termine,EMP-7240,4556
2,100002,ALU-T2-18,Saint-Etienne,3,2026-04-11T09:54:06,2026-04-12T16:54:06,suspendu,EMP-1939,1308
3,100003,ALU-T1-22,Roubaix,1,2026-04-20T22:33:08,2026-04-22T04:33:08,termine,EMP-3531,2968
4,100004,ALU-T2-25,Roubaix,4,2026-04-24T01:03:02,2026-04-25T21:03:02,termine,EMP-8778,3278


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ordre_id,2000.0,NaN,NaN,NaN,100999.5,577.494589,100000.0,100499.75,100999.5,101499.25,101999.0
produit_ref,2000,10,INOX-316-4,227,NaN,NaN,NaN,NaN,NaN,NaN,NaN
site,2000,2,Roubaix,1108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line_id,2000.0,NaN,NaN,NaN,2.316,1.033769,1.0,1.0,2.0,3.0,4.0
date_lancement,2000,1999,2026-04-29T19:55:27,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_fin_prevue,2000,1999,2026-04-14T22:51:14,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
statut,2000,4,termine,1559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ouvrier_id,1891,1689,EMP-4182,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
quantite_kg,2000.0,NaN,NaN,NaN,2528.689,1425.269007,51.0,1270.25,2534.0,3739.25,4999.0


Lignes: 2,000
Colonnes: 9
Mémoire estimée en RAM: 0.74 Mo


,dtype,missing,missing_pct,nunique
ouvrier_id,object,109,5.45,1689
site,object,0,0.00,2
line_id,int64,0,0.00,4
statut,object,0,0.00,4
produit_ref,object,0,0.00,10
quantite_kg,int64,0,0.00,1624
date_lancement,object,0,0.00,1999
date_fin_prevue,object,0,0.00,1999
ordre_id,int64,0,0.00,2000



Répartition de statut:


statut
termine     1559
en_cours     197
suspendu     139
annule       105
Name: count, dtype: int64

Colonnes potentiellement sensibles: ['ordre_id', 'line_id', 'ouvrier_id']
Taux de manquants sur ouvrier_id: 5.45%


> **Observations** :
>
> - Volume : **2 000 lignes**, 9 colonnes, ~**0,74 Mo** en RAM.
> - Qualité observée : `ouvrier_id` a **109 manquants (5,45%)** ; dates actuellement en texte (`object`).
> - Répartition statut : `termine` majoritaire (**1559**), puis `en_cours` (197), `suspendu` (139), `annule` (105).
> - ⚠️ Risque RGPD : `ouvrier_id` est une donnée de personnel (identifiant pseudonymisé mais traçable). `ordre_id` peut aussi devenir sensible en recoupement.
> - Pertinence métier : source structurée de référence pour le cycle des ordres de production.
> - Question pour Sébastien : quel est le motif des `ouvrier_id` manquants (absence d’affectation, qualité de saisie, anonymisation partielle) ?

## Source 3 — Logs machines (texte)

In [6]:
log_path = DATA_DIR / "logs_machines.log"

# Lecture des logs dans pandas sans transformation de fond
df_logs = pd.read_table(log_path, header=None, names=["raw_line"], dtype="string")
df_logs.info()

display(df_logs.head())
display(df_logs.describe(include="all").transpose())

# Volume
print(f"Lignes: {len(df_logs):,}")
print(f"Colonnes: {df_logs.shape[1]}")
print(f"Taille fichier disque: {log_path.stat().st_size / 1024:.1f} Ko")
mem_mb_logs = df_logs.memory_usage(deep=True).sum() / (1024**2)
print(f"Mémoire estimée en RAM: {mem_mb_logs:.2f} Mo")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   raw_line  30000 non-null  string
dtypes: string(1)
memory usage: 234.5 KB


,raw_line
0,[2026-04-01T00:00:16] Lyon LINE-1 INFO: shift_...
1,[2026-04-01T00:01:07] Saint-Etienne LINE-2 INF...
2,[2026-04-01T00:01:34] Saint-Etienne LINE-3 ERR...
3,[2026-04-01T00:04:18] Roubaix LINE-4 INFO: mai...
4,[2026-04-01T00:04:35] Lyon LINE-1 INFO: toolin...


,count,unique,top,freq
raw_line,30000,29999,[2026-04-14T08:49:35] Roubaix LINE-1 INFO: mai...,2


Lignes: 30,000
Colonnes: 1
Taille fichier disque: 1872.8 Ko
Mémoire estimée en RAM: 3.20 Mo


In [8]:
# Défauts manifestes
print(f"Lignes vides: {(df_logs['raw_line'].isna().sum() + (df_logs['raw_line'].fillna('').str.strip() == '').sum()):,}")
print(f"Doublons exacts de ligne: {df_logs.duplicated().sum():,}")

# Types d'événements (INFO/WARN/ERROR)
event_type = df_logs["raw_line"].str.extract(r"\b(INFO|WARN|ERROR)\b", expand=False)
display(event_type.value_counts(dropna=False).rename_axis("event_type").to_frame("count"))

# Signaux RGPD potentiels dans le texte (heuristiques)
flags = pd.DataFrame({
    "has_email": df_logs["raw_line"].str.contains(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", regex=True, na=False),
    "has_ipv4": df_logs["raw_line"].str.contains(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", regex=True, na=False),
    "has_phone": df_logs["raw_line"].str.contains(r"\b(?:\+33|0)[1-9](?:[ .-]?\d{2}){4}\b", regex=True, na=False),
    "has_user_token": df_logs["raw_line"].str.contains(r"\b(?:user|ouvrier|employee|operator|login|token|session|id)\b", regex=True, case=False, na=False),
})
display(flags.sum().to_frame("nb_lignes_signal"))

# Exemples de lignes à risque potentiel
risk_mask = flags.any(axis=1)
display(df_logs.loc[risk_mask, ["raw_line"]].head(10))

Lignes vides: 0
Doublons exacts de ligne: 1


,count
event_type,
INFO,22501
WARN,5758
ERROR,1741


,nb_lignes_signal
has_email,0
has_ipv4,0
has_phone,0
has_user_token,425


,raw_line
5,[2026-04-01T00:04:54] Roubaix LINE-3 ERROR: em...
33,[2026-04-01T00:23:28] Saint-Etienne LINE-3 ERR...
83,[2026-04-01T01:32:17] Saint-Etienne LINE-1 ERR...
95,[2026-04-01T01:45:21] Roubaix LINE-3 ERROR: em...
117,[2026-04-01T02:13:15] Roubaix LINE-1 ERROR: em...
227,[2026-04-01T05:04:19] Roubaix LINE-4 ERROR: em...
234,[2026-04-01T05:28:42] Saint-Etienne LINE-1 ERR...
285,[2026-04-01T06:42:21] Roubaix LINE-4 ERROR: em...
309,[2026-04-01T07:17:12] Saint-Etienne LINE-1 ERR...
363,[2026-04-01T08:40:36] Saint-Etienne LINE-3 ERR...


> **Observations** :
>
> - Volume : **30 000 lignes**, 1 colonne (`raw_line`), fichier ~**1,83 Mo** sur disque.
> - Format : texte semi-structuré ; parse regex nécessaire pour extraction fiable des attributs (timestamp, site, line, niveau, message).
> - Qualité observée : **0 ligne vide**, **1 doublon** ; distribution événements = INFO 22 501, WARN 5 758, ERROR 1 741.
> - Risques RGPD : pas d’email/téléphone/IP détectés, mais **425 lignes** contiennent des marqueurs d’identifiants (`employee`, `user`, `id`, etc.).
> - Croisement avec les capteurs IoT : faisable par timestamp/site/ligne pour investiguer les incidents (ex. pics ERROR vs dérive vibration/temperature à Roubaix).
> - Question pour Sébastien : le format de log est-il contractuel/stable ou susceptible d’évoluer selon les versions machine ?

## Synthèse pour `identification_sources.md`

Remplis le tableau d'inventaire dans `../identification_sources.md` à
partir des observations ci-dessus.